In [ ]:
!pip install -U \
    sentence-transformers==2.2.2 \
    transformers==4.30.2 \
    huggingface-hub==0.15.1 \
    accelerate==0.20.3 \
    torch==2.0.1

In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch
import pandas as pd

In [ ]:
df = pd.read_csv('/kaggle/input/previous-labeled-topics/updated_predictions.csv')

# Load the stable model
model = SentenceTransformer("allenai-specter")

# Encode
topic_emb = model.encode(df["topic_words"].tolist(), convert_to_tensor=True)
gpt_emb = model.encode(df["topic_label"].tolist(), convert_to_tensor=True)
small_emb = model.encode(df["predicted_label_small"].tolist(), convert_to_tensor=True)
base_emb = model.encode(df["predicted_label_base"].tolist(), convert_to_tensor=True)
large_emb = model.encode(df["predicted_label_large"].tolist(), convert_to_tensor=True)

# Compute cosine similarity diagonally
df["similarity_gpt"] = torch.diag(util.cos_sim(topic_emb, gpt_emb)).cpu().numpy()
df["similarity_small"] = torch.diag(util.cos_sim(topic_emb, small_emb)).cpu().numpy()
df["similarity_base"] = torch.diag(util.cos_sim(topic_emb, base_emb)).cpu().numpy()
df["similarity_large"] = torch.diag(util.cos_sim(topic_emb, large_emb)).cpu().numpy()

df.head()

In [ ]:
import matplotlib.pyplot as plt

# Sort the DataFrame
df_sorted = df.sort_values(by="similarity_gpt").reset_index(drop=True)

similarity_cols = ["similarity_small", "similarity_base", "similarity_large", "similarity_gpt"]

plt.figure(figsize=(12, 6))
for col in similarity_cols:
    plt.plot(df_sorted.index, df_sorted[col], label=col, linewidth=2)

plt.ylim(bottom=0.6)

plt.title("Semantic Similarity (Sorted by similarity_gpt)", fontsize=14)
plt.xlabel("Entry Index")
plt.ylabel("Cosine Similarity")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
df[["similarity_small", "similarity_base", "similarity_large", "similarity_gpt"]].describe()

In [ ]:
import pandas as pd

cols = ["similarity_small", "similarity_base", "similarity_large"]

results = []

for col in cols:
    diff = (df["similarity_gpt"] - df[col]).abs()  # absolute difference
    above = (df[col] > df["similarity_gpt"]).sum()
    below = (df[col] < df["similarity_gpt"]).sum()
    total = len(df)
    
    results.append({
        "Model": col,
        "Above GPT": f"{above} ({above/total:.1%})",
        "Below GPT": f"{below} ({below/total:.1%})",
        "Mean Δ": f"{diff.mean():.4f}",
        "Max Δ": f"{diff.max():.4f}"
    })

# Convert to DataFrame
summary_df = pd.DataFrame(results)

# Display nicely
from IPython.display import display
display(summary_df.style.set_caption("Similarity Comparison vs GPT")
                       .background_gradient(subset=["Mean Δ", "Max Δ"], cmap="Blues")
                       .set_table_styles([
                           {'selector': 'caption', 'props': [('font-size', '16px'),
                                                             ('font-weight', 'bold'),
                                                             ('text-align', 'center')]}
                       ])
                       .hide(axis="index"))


In [ ]:
df.to_csv("/kaggle/working/predictions_similarity.csv", index=False)